In [17]:
# backend/
#   app/
#     main.py                     # FastAPI app entry point
#     api/
#       routes_games.py           # REST endpoints (create game, action, get state)
#       ws_games.py               # WebSocket endpoint for realtime updates
#     schemas/                    # Pydantic models (contracts)
#       card_defs.py              # CardDef, PlayDef (loaded from JSON/YAML)
#       actions.py                # Client->server action models
#       responses.py              # Server->client response models (DTOs)
#     services/
#       game_service.py           # Orchestrates: load state, apply engine, save, broadcast
#       card_catalog.py           # Loads + validates cards from files
#     engine/                     # Pure game logic (no web, no DB)
#       state.py                  # dataclasses: GameState, PlayerState, etc.
#       rules.py                  # apply_action(), validation, turn rules
#       effects/
#         registry.py             # effect_name -> function mapping
#         draw.py
#         rent.py
#         steal.py
#         payment.py              # your payment logic lives here
#     db/
#       models.py                 # SQLAlchemy models (Game row, Move row)
#       session.py                # DB session / engine
#       repo.py                   # DB helper functions (get_game, save_game, log_move)
#   cards/
#     base/                       # official card definitions (JSON)
#     custom/                     # your custom cards / mod cards (JSON)
#   tests/
#     test_payment.py
#     test_draw.py
#     test_rent.py
#   pyproject.toml (or requirements.txt)


In [18]:
## Pydantic Models for Cards and Raw Cards

from pydantic import BaseModel, Field, ConfigDict
from typing import Any, Dict, List, Optional, Literal

class RawEffect(BaseModel):
    type: str
    model_config = ConfigDict(extra="allow")  # keep other keys

class RawCard(BaseModel):
    id: str
    name: str
    type: str                 # your JSON uses "type"
    count: int
    bank_value: int

    colors: Optional[List[str]] = None
    property_group: Optional[str] = None
    effect: Optional[RawEffect] = None
    rent_by_count: Optional[List[int]] = None

    bankable: Optional[bool] = None
    image_url: Optional[str] = None
    restrictions: Optional[Dict[str, Any]] = None
    note: Optional[str] = None

    model_config = ConfigDict(extra="allow")  # keep any future fields

class PlayDef(BaseModel):
    effect: str
    params: Dict[str, Any] = Field(default_factory=dict)

class CardDef(BaseModel):
    id: str
    name: str
    kind: Literal["money","property","property_wild","rent","action"]
    copies: int
    money_value: int
    colors: Optional[List[str]] = None
    rent_by_count: Optional[List[int]] = None 
    play: Optional[PlayDef] = None
    meta: Dict[str, Any] = Field(default_factory=dict)


In [19]:
## Convert RawCard to CardDef

#Convert RawCard to CardDef
def raw_to_carddef(raw: RawCard) -> Optional[CardDef]:
    kind_map = {
        "money": "money",
        "property": "property",
        "property_wild": "property_wild",
        "rent": "rent",
        "action": "action"
    }
    
    kind = kind_map.get(raw.type)
    if not kind:
        return None  # Unknown type
    
    play_def = None
    if raw.effect:
        play_def = PlayDef(effect=raw.effect.type, params={k: v for k, v in raw.effect.model_dump().items() if k != "type"})
    
    card_def = CardDef(
        id=raw.id,
        name=raw.name,
        kind=kind,
        copies=raw.count,
        money_value=raw.bank_value,
        rent_by_count=raw.rent_by_count,
        colors=raw.colors,
        play=play_def,
        meta={
            "property_group": raw.property_group,
            "bankable": raw.bankable,
            "image_url": raw.image_url,
            "restrictions": raw.restrictions,
            "note": raw.note
        }
    )
    
    return card_def

In [20]:
## Load all cards

from pathlib import Path
import json
from typing import Dict, List

def load_card_defs_from_dir(cards_dir: str) -> Dict[str, CardDef]:
    catalog: Dict[str,CardDef] = {}

    for path in Path(cards_dir).glob("*.json"):
        raw_data = json.loads(path.read_text(encoding="utf-8"))

        # Normalize to a list
        if isinstance(raw_data, list):
            raw_cards = [RawCard.model_validate(item) for item in raw_data]
        else:
            raw_cards = [RawCard.model_validate(raw_data)]

        for rc in raw_cards:
            cd = raw_to_carddef(rc)
            if cd is None:
                continue
            if cd.id in catalog:
                raise ValueError(f"Duplicate card ID found: {cd.id}")
            catalog[cd.id] = cd

    return catalog

In [21]:
## Build Rent Table from catalog (This will be done once per game)
import os

def build_rent_table(catalog: Dict[str, CardDef]) -> Dict[str, List[int]]:
    rent_table: Dict[str, List[int]] = {}
    for card_def in catalog.values():
        if card_def.kind == "property" and card_def.meta.get("property_group") and card_def.rent_by_count:
            # All properties in a group should have same rent_by_count
            rent_table[card_def.meta["property_group"]] = card_def.rent_by_count
    return rent_table

# Pathing here:
base_path = os.getcwd()

all_cards_json_path = os.path.join(base_path, "all_cards.json")

all_cards_json = json.load(open(all_cards_json_path, "r"))
cards_path = os.path.join(base_path,"backend" ,"cards", "base")

#Usage
catalog = load_card_defs_from_dir(cards_path)

RENT_TABLE = build_rent_table(catalog)

In [ ]:
## GameState and PlayerState and DeckState:

from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any, Literal


class DeckState(BaseModel):
    draw_pile: List[str] = Field(default_factory = list)
    discard_pile: List[str] = Field(default_factory = list)

class PlayerState(BaseModel):
    id: str
    hand: List[str] = Field(default_factory=list)        # card ids
    bank: List[str] = Field(default_factory=list)        # card ids
    properties: Dict[str, List[str]] = Field(default_factory=dict)  # color -> card ids
    buildings: Dict[str, List[str]] = Field(default_factory=dict)  # color -> building card ids

class GameState(BaseModel):
    id: str
    players: Dict[str, PlayerState]
    deck: DeckState
    current_player_id: Optional[str] = None
    turn_number: int = 1
    actions_taken: int = 0
    pending_actions: Dict[str, Dict[str, Any]] = Field(default_factory=dict)

## API Action and Response Models

class ChangeWildPayload(BaseModel):
    card_id: str
    new_color: str

class ActionRequest(BaseModel):
    action_type: Literal[
        "play_bank",
        "play_property",
        "change_wild",
        "discard",
        "end_turn",
        "play_action_counterable",
        "play_action_non_counterable",
    ]
    player_id: str

    # card ids
    card_id: Optional[str] = None
    bank_card_id: Optional[str] = None
    property_card_id: Optional[str] = None

    # property placement
    property_color: Optional[str] = None

    # rent
    rent_color: Optional[str] = None
    double_rent_ids: List[str] = Field(default_factory=list)

    # targeting
    target_player_id: Optional[str] = None

    # property manipulation
    steal_card_id: Optional[str] = None
    give_card_id: Optional[str] = None
    steal_color: Optional[str] = None

    # change wild
    change_wild: Optional[ChangeWildPayload] = None

    # discard
    discard_ids: List[str] = Field(default_factory=list)

class PaymentTarget(BaseModel):
    player_id: str
    amount: int

class PaymentRequired(BaseModel):
    request_id: str
    receiver_id: str
    targets: List[PaymentTarget]

class PendingPrompt(BaseModel):
    pending_id: str
    target_player: str
    prompt: str

class ResponseRequired(BaseModel):
    pending_requests: List[PendingPrompt]

class ActionResponse(BaseModel):
    status: Literal["ok", "error"]
    response_type: Literal["action_resolved", "payment_required", "response_required"]
    state: Optional[GameState] = None
    payment_request: Optional[PaymentRequired] = None
    response_required: Optional[ResponseRequired] = None
    message: Optional[str] = None
    log: Optional[Dict[str, Any]] = None

class PaymentRequest(BaseModel):
    request_id: str
    payer_id: str
    receiver_id: str
    bank: List[str] = Field(default_factory=list)
    properties: List[str] = Field(default_factory=list)
    buildings: List[str] = Field(default_factory=list)

class PaymentResponse(BaseModel):
    status: Literal["ok", "error"]
    response_type: Literal["payment_applied"]
    state: Optional[GameState] = None
    message: Optional[str] = None
    log: Optional[Dict[str, Any]] = None



class PendingResponseRequest(BaseModel):
    pending_id: str
    player_id: str
    response: Literal["accept", "just_say_no"]


In [23]:
## Build Deck Function

import random
from typing import Dict, List

def build_deck(catalog: Dict[str, CardDef], seed = 42) -> List[str]:
    deck: List[str] = []

    #catalog is key: card id, value: CardDef
    for cd in catalog.values():
        deck.extend([cd.id] * cd.copies)
    
    #Shuffle deck with seed
    random.seed(seed)
    random.shuffle(deck)

    #Check if deck is 110 cards long

    if len(deck) != 106: #110 - 4 helper cards
        
        raise ValueError(f"Deck size is {len(deck)}, expected 110.")
    
    return deck


#Test this function out

catalog = load_card_defs_from_dir(cards_path)
deck = build_deck(catalog, seed=123)


In [24]:
## Create New Game:
import uuid
from typing import Dict, List
#from .engine.state import GameState, PlayerState, DeckState #TODO: we will add this later
#from .services.card_catalog import build_deck #TODO: we will also add this later

STARTING_HAND_SIZE = 5

def create_new_game(player_ids: List[str], 
                    catalog: Dict[str, CardDef]) -> GameState:
    # 1) Build and shuffle deck
    draw_pile = build_deck(catalog)
    deck = DeckState(draw_pile=draw_pile, discard_pile=[])

    # 2) Create players
    players = {pid: PlayerState(id=pid) for pid in player_ids}

    # 3) Deal starting hands
    for _ in range(STARTING_HAND_SIZE):
        for pid in player_ids:
            if not deck.draw_pile:
                raise ValueError("Deck is empty while dealing starting hands.")
            card_id = deck.draw_pile.pop(0)
            players[pid].hand.append(card_id)

    # 4) Create game state
    return GameState(
        id=str(uuid.uuid4()),
        players=players,
        deck=deck,
        current_player_id=player_ids[0] if player_ids else None,
        turn_number=1,
    )

In [25]:
## Charge Rent and building bonus rent calculation!
from collections import Counter
from typing import Dict, List, Optional

# Expect this to exist once you load cards:
# RENT_TABLE = build_rent_table(raw_cards_or_catalog)

def charge_rent_amount(
    state: GameState,
    catalog: Dict[str, CardDef],
    player_id: str,
    rent_card_id: str,
    color: str,
    double_rent_ids: Optional[List[str]] = None,
    require_in_hand: bool = True
) -> int:
    
    if "RENT_TABLE" not in globals():
        raise ValueError("RENT_TABLE is not defined. Build it once before charging rent.")
    rent_table = RENT_TABLE

    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")
    player = state.players[player_id]

    # Ensure player has the rent card + any double rent cards
    if require_in_hand:
        hand_counts = Counter(player.hand)
        play_ids = [rent_card_id] + (double_rent_ids or [])
        play_counts = Counter(play_ids)
        for cid, cnt in play_counts.items():
            if cnt > hand_counts.get(cid, 0):
                raise ValueError(f"Player does not have {cnt} copies of {cid} in hand.")

    # Validate rent card
    if rent_card_id not in catalog:
        raise ValueError(f"Unknown card id: {rent_card_id}")
    rent_card = catalog[rent_card_id]
    if rent_card.kind != "rent":
        raise ValueError("Card is not a rent card.")

    # Validate color choice
    allowed_colors = rent_card.colors or []
    if "any" not in allowed_colors and color not in allowed_colors:
        raise ValueError(f"Rent card cannot be used for color '{color}'.")

    # Base rent from RENT_TABLE
    if color not in rent_table:
        raise ValueError(f"No rent table defined for color '{color}'.")
    rent_steps = rent_table[color]
    if not rent_steps:
        raise ValueError(f"Rent table for '{color}' is empty.")

    prop_count = len(player.properties.get(color, []))
    if prop_count == 0:
        raise ValueError(f"Player has no properties in color '{color}'.")
    step_index = min(prop_count, len(rent_steps)) - 1
    base_rent = rent_steps[step_index]

    #Apply Building bonus only on full sets
    # only allow building bonus if full set
    full_set = prop_count >= len(rent_steps)
    bonus = building_bonus_for_color(state, catalog, player_id, color) if full_set else 0


    # Apply Double The Rent modifiers (stackable)
    multiplier = 1
    for dr_id in (double_rent_ids or []):
        if dr_id not in catalog:
            raise ValueError(f"Unknown card id: {dr_id}")
        dr = catalog[dr_id]

        if dr.kind != "action" or not dr.play:
            raise ValueError(f"{dr_id} is not a valid Double The Rent card.")
        
        if dr.play.effect != "modifier" or dr.play.params.get("applies_to") != "rent":
            raise ValueError(f"{dr_id} is not a valid rent modifier.")
        multiplier *= int(dr.play.params.get("multiplier", 2))

    return (base_rent+bonus) * multiplier


#Add extra rent if you have a building in a appropriate full property set
#Can only have houses and then hotels on a full property set
#No houses or hotels on railroads or utilities

def building_bonus_for_color(
    state: GameState,
    catalog: Dict[str, CardDef],
    player_id: str,
    color: str,
) -> int:
    bonus = 0
    for bid in state.players[player_id].buildings.get(color, []):
        cd = catalog[bid]
        if cd.play and cd.play.effect == "building":
            bonus += int(cd.play.params.get("rent_bonus", 0))
    return bonus


In [26]:
## Use Action

def use_action(state: GameState, max_actions: int = 3) -> None:
    if state.actions_taken >= max_actions:
        raise ValueError("No actions remaining this turn.")
    state.actions_taken += 1

In [ ]:
## Play Bank

from typing import Dict


def play_bank(
    state: GameState,
    player_id: str,
    card_id: str,
    catalog: Dict[str, CardDef],
) -> GameState:
    
    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    player = state.players[player_id]

    if card_id not in player.hand:
        raise ValueError("Card not in player hand.")

    if card_id not in catalog:
        raise ValueError(f"Unknown card id: {card_id}")

    cd = catalog[card_id]

    # In Monopoly Deal, any card can be banked for its money value
    if cd.money_value <= 0 and cd.kind == "property": #Can't bank property cards since while they have money value, they are properties
        raise ValueError("This card has no money value or it's a property and cannot be banked.") #For example this can be a wild multiproperty card

    # Move from hand -> bank
    player.hand.remove(card_id)
    player.bank.append(card_id)

    return None



In [ ]:
## Play Property

from typing import Dict, Optional
# from .state import GameState
# from ..schemas.card_defs import CardDef  # adjust path if needed #TODO: Add this file structure later

def play_property(state: GameState,
                  catalog: Dict[str, CardDef],
                  player_id: str,
                  card_id: str,
                  color_if_wild: Optional[str] = None) -> GameState:

    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    player = state.players[player_id]

    if card_id not in player.hand:
        raise ValueError("Card not in player hand.")

    if card_id not in catalog:
        raise ValueError(f"Unknown card id: {card_id}")

    cd = catalog[card_id]

    if cd.kind not in {"property", "property_wild"}:
        raise ValueError("Card is not a property or a wild property.")

    if not cd.colors:
        raise ValueError("Property card has no colors defined.")

    # Choose color
    if cd.kind == "property":
        # If only one color, auto-use it unless a different color was passed

        if len(cd.colors) == 1:
            chosen_color = cd.colors[0]
            if color_if_wild and color_if_wild != chosen_color:
                raise ValueError(f"Invalid color '{color_if_wild}' for this property.")
        else:
            # Multi-color property behaves like wild; require explicit choice
            if not color_if_wild or color_if_wild not in cd.colors:
                raise ValueError("Must choose a valid color for this property.")
            chosen_color = color_if_wild
    else:
        # property_wild: must choose one of its colors
        if not color_if_wild or color_if_wild not in cd.colors:
            raise ValueError("Must choose a valid color for a wild property.")
        chosen_color = color_if_wild

    # Move card to properties
    player.hand.remove(card_id)
    player.properties.setdefault(chosen_color, []).append(card_id)

    return None
    
    


In [29]:
def play_building(
    state: GameState,
    catalog: Dict[str, CardDef],
    player_id: str,
    card_id: str,
    color: str,
) -> None:
    player = state.players[player_id]
    if card_id not in player.hand:
        raise ValueError("Building card not in hand.")
    cd = catalog[card_id]

    if not cd.play or cd.play.effect != "building":
        raise ValueError("Card is not a building action.")

    # disallowed groups
    disallowed = set(cd.play.params.get("disallowed_groups", []))
    if color in disallowed:
        raise ValueError("Cannot place building on this color.")

    # must have full set
    needed = len(RENT_TABLE[color])
    if len(player.properties.get(color, [])) < needed:
        raise ValueError("Need full set to place building.")

    # hotel requires house
    if cd.play.params.get("requires_house"):
        if "action_house" not in player.buildings.get(color, []):
            raise ValueError("Hotel requires a house on this set.")

    # move card from hand → buildings
    player.hand.remove(card_id)
    player.buildings.setdefault(color, []).append(card_id)


In [30]:
## Draw Cards

import random
from typing import List
#from .state import GameState #TODO: Add this file structure later

def draw_cards(state: GameState, player_id: str, n: int = 1) -> GameState:
    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    for _ in range(n):
        if not state.deck.draw_pile:
            if not state.deck.discard_pile:
                raise ValueError("Cannot draw: no cards left in draw or discard pile.")
            # move discard into draw and shuffle
            state.deck.draw_pile = state.deck.discard_pile
            state.deck.discard_pile = []
            random.shuffle(state.deck.draw_pile)

        card_id = state.deck.draw_pile.pop(0)
        state.players[player_id].hand.append(card_id)

    return state

In [31]:
## Discard Cards
from collections.abc import Iterable

def discard_cards(state: GameState, 
                  player_id: str, 
                  card_ids: Iterable[str]) -> GameState:
    
    """"
    Card_ids function argument is the card ids to be discarded from the player's hand that the player chooses to discard.
    We take this directly from the UI and input it into the engine to update the game state.
    """
    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    hand = state.players[player_id].hand
    for cid in card_ids:
        if cid not in hand:
            raise ValueError(f"Player {player_id} does not have card {cid} in hand.")
        hand.remove(cid)
        state.deck.discard_pile.append(cid)

    return state


In [32]:
# Discard Action Cards
def discard_action_cards(state: GameState, actor: PlayerState, card_ids: List[str]) -> None:
    for cid in card_ids:
        if cid not in actor.hand:
            raise ValueError(f"Card {cid} not in hand.")
        actor.hand.remove(cid)
        state.deck.discard_pile.append(cid)

In [33]:
#End Turn

from typing import List
#from .state import GameState #TODO: Add this file structure later

def end_turn(state: GameState, turn_order: List[str]) -> GameState:

    if not turn_order:
        raise ValueError("turn_order is empty.")
    if state.current_player_id not in turn_order:
        raise ValueError("current_player_id not found in turn_order.")

    idx = turn_order.index(state.current_player_id)
    next_idx = (idx + 1) % len(turn_order)

    state.current_player_id = turn_order[next_idx]
    state.turn_number += 1
    state.actions_taken = 0
    
    return state

In [40]:
## Change Wild Color

def change_wild_color(
    state: GameState,
    catalog: Dict[str, CardDef],
    player_id: str,
    card_id: str,
    new_color: str,
) -> GameState:
    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    player = state.players[player_id]

    # Find current color pile containing this card
    current_color = None
    for color, cards in player.properties.items():
        if card_id in cards:
            current_color = color
            break
    if current_color is None:
        raise ValueError("Card not found in player properties.")

    if card_id not in catalog:
        raise ValueError(f"Unknown card id: {card_id}")
    cd = catalog[card_id] 

    if cd.kind != "property_wild":
        raise ValueError("Card is not a wild property.")
    if not cd.colors:
        raise ValueError("Card has no color options.")

    # Validate color choice
    if "any" in cd.colors:
        # optional: only allow real property colors
        if "RENT_TABLE" in globals() and new_color not in RENT_TABLE:
            raise ValueError(f"Invalid color '{new_color}'.")
    else:
        if new_color not in cd.colors:
            raise ValueError(f"Card cannot be set to color '{new_color}'.")

    # Move card to the new color pile
    player.properties[current_color].remove(card_id)
    player.properties.setdefault(new_color, []).append(card_id)

    return state

In [35]:
## Process Property Manipulation

from typing import Dict, Optional, Any

def get_set_size(color: str) -> int:
    if "RENT_TABLE" not in globals():
        raise ValueError("RENT_TABLE is not defined. Build it once before property actions.")
    if color not in RENT_TABLE:
        raise ValueError(f"No rent table for color '{color}'.")
    return len(RENT_TABLE[color])

def is_full_set(player: PlayerState, color: str) -> bool:
    return len(player.properties.get(color, [])) >= get_set_size(color)

def find_card_color(properties: Dict[str, List[str]], card_id: str) -> str:
    for color, cards in properties.items():
        if card_id in cards:
            return color
    raise ValueError(f"Card id {card_id} not found in properties.")

def process_property_manipulation(
    state: GameState,
    catalog: Dict[str, CardDef],
    actor_id: str,
    action_card_id: str,
    target_player_id: str,
    *,
    steal_card_id: Optional[str] = None,   # Sly Deal & Forced Deal
    give_card_id: Optional[str] = None,    # Forced Deal
    steal_color: Optional[str] = None,     # Deal Breaker (full set)
) -> Dict[str, Any]:
    
    if actor_id not in state.players:
        raise ValueError(f"Unknown actor_id: {actor_id}")
    if target_player_id not in state.players:
        raise ValueError(f"Unknown target_player_id: {target_player_id}")
    if actor_id == target_player_id:
        raise ValueError("Cannot target yourself.")

    actor = state.players[actor_id]
    target = state.players[target_player_id]

    if action_card_id not in catalog:
        raise ValueError(f"Unknown card id: {action_card_id}")

    action_card = catalog[action_card_id]
    if action_card.kind != "action" or not action_card.play:
        raise ValueError("Card is not a playable action.")

    effect = action_card.play.effect
    params = action_card.play.params


    ## Deal Breaker Logic

    if effect == "steal_full_set":
        if not steal_color:
            raise ValueError("steal_color is required for Deal Breaker.")
        if not is_full_set(target, steal_color):
            raise ValueError(f"Target does not have a full set of '{steal_color}'.")

        stolen_cards = target.properties.get(steal_color, [])
        target.properties[steal_color] = [] #The target loses the full set
        actor.properties.setdefault(steal_color, []).extend(stolen_cards) #The actor gains the full set

        #Handle any building transfers if needed:
        if params.get("includes_buildings"):
            b = target.buildings.get(steal_color, [])
            target.buildings[steal_color] = []
            actor.buildings.setdefault(steal_color, []).extend(b)

        return {
            "type": "steal_full_set",
            "actor_id": actor_id,
            "target_id": target_player_id,
            "color": steal_color,
            "cards": stolen_cards,
        }
    ## Sly Deal Logic

    if effect == "steal_property":
        if not steal_card_id:
            raise ValueError("steal_card_id is required for Sly Deal.")
        steal_color_actual = find_card_color(target.properties, steal_card_id)

        if not params.get("from_full_set_allowed", False) and is_full_set(target, steal_color_actual):
            raise ValueError("Cannot steal from a full set.")

        target.properties[steal_color_actual].remove(steal_card_id)
        actor.properties.setdefault(steal_color_actual, []).append(steal_card_id)

        return {
            "type": "steal_property",
            "actor_id": actor_id,
            "target_id": target_player_id,
            "card_id": steal_card_id,
            "color": steal_color_actual,
        }
    ## Forced Deal Logic
    
    if effect == "swap_property":
        if not steal_card_id or not give_card_id:
            raise ValueError("steal_card_id and give_card_id are required for Forced Deal.")

        steal_color_actual = find_card_color(target.properties, steal_card_id)
        give_color_actual = find_card_color(actor.properties, give_card_id)

        if not params.get("from_full_set_allowed", False):
            if is_full_set(target, steal_color_actual):
                raise ValueError("Cannot take from target full set.")
            if is_full_set(actor, give_color_actual):
                raise ValueError("Cannot give from your full set.")

        target.properties[steal_color_actual].remove(steal_card_id)
        actor.properties[give_color_actual].remove(give_card_id)

        actor.properties.setdefault(steal_color_actual, []).append(steal_card_id)
        target.properties.setdefault(give_color_actual, []).append(give_card_id)

        return {
            "type": "swap_property",
            "actor_id": actor_id,
            "target_id": target_player_id,
            "stolen_card_id": steal_card_id,
            "given_card_id": give_card_id,
            "stolen_color": steal_color_actual,
            "given_color": give_color_actual,
        }

    raise ValueError(f"Unsupported property manipulation effect: {effect}")


In [ ]:
## Is_Counterable and Pending helpers

import uuid
from typing import Dict, Any, List, Optional

JSN_CARD_ID = "action_just_say_no"

def is_counterable(card: CardDef) -> bool:
    if card.kind == "rent":
        return True
    if card.kind == "action" and card.play:
        return card.play.effect in {"charge_players", "steal_full_set", "steal_property", "swap_property"}
    return False

def create_pending_action(state: GameState, *, source_player: str, target_player: str,
                          card_id: str, card_kind: str, effect: Optional[str],
                          payload: Dict[str, Any]) -> str:
    pending_id = f"pend_{uuid.uuid4().hex}"
    state.pending_actions[pending_id] = {
        "id": pending_id,
        "source_player": source_player,
        "target_player": target_player,
        "awaiting_player": target_player,
        "card_id": card_id,
        "card_kind": card_kind,
        "effect": effect,
        "payload": payload,
        "jsn_count": 0,
    }
    return pending_id



In [ ]:
## Apply Action Effects 

def apply_action_effects(
    state: GameState,
    catalog: Dict[str, CardDef],
    *,
    player_id: str,
    card_id: str,
    rent_color: Optional[str] = None,
    double_rent_ids: Optional[List[str]] = None,
    target_player_id: Optional[str] = None,
    steal_card_id: Optional[str] = None,
    give_card_id: Optional[str] = None,
    steal_color: Optional[str] = None,
    already_discarded: bool = False,
) -> Dict[str, Any]:
    
    #Define Actor, Card, and Effect for later logic
    actor = state.players[player_id]
    card = catalog[card_id]
    effect = card.play.effect if card.play else None

    # discard action card here only if not already discarded
    if not already_discarded and effect != "building":
        discard_action_cards(state, actor, [card_id] + (double_rent_ids or []))

    # Rent
    if card.kind == "rent":
        amount = charge_rent_amount(
            state=state,
            catalog=catalog,
            player_id=player_id,
            rent_card_id=card_id,
            color=rent_color,
            double_rent_ids=double_rent_ids,
            require_in_hand= not already_discarded
        )
        return {
            "status": "ok",
            "response_type": "payment_required",
            "state": state,
            "payment_request": {
                "request_id": f"pay_{uuid.uuid4().hex}",
                "receiver_id": player_id,
                "targets": [{"player_id": target_player_id, "amount": amount}],
            },
            "log": {"action": "rent", "player_id": player_id, "amount": amount},
        }

    # Action cards
    if card.kind == "action":

        if effect == "draw_cards":
            amount = int(card.play.params.get("amount", 0))
            draw_cards(state, player_id, amount)
            return {
                "status": "ok",
                "response_type": "action_resolved",
                "state": state,
                "log": {"action": "draw_cards", "player_id": player_id, "amount": amount},
            }

        if effect in {"steal_full_set", "steal_property", "swap_property"}:
            result = process_property_manipulation(
                state=state,
                catalog=catalog,
                actor_id=player_id,
                action_card_id=card_id,
                target_player_id=target_player_id,
                steal_card_id=steal_card_id,
                give_card_id=give_card_id,
                steal_color=steal_color,
            )
            return {
                "status": "ok",
                "response_type": "action_resolved",
                "state": state,
                "log": result,
            }

        if effect == "charge_players":
            amount = int(card.play.params.get("amount", 0))
            return {
                "status": "ok",
                "response_type": "payment_required",
                "state": state,
                "payment_request": {
                    "request_id": f"pay_{uuid.uuid4().hex}",
                    "receiver_id": player_id,
                    "targets": [{"player_id": target_player_id, "amount": amount}],
                },
                "log": {"action": "charge_players", "player_id": player_id, "amount": amount},
            }
        
        if effect == "building":
            play_building(state, catalog, player_id, card_id, rent_color)  # use rent_color as chosen set color
            return {
            "status": "ok",
            "response_type": "action_resolved",
            "state": state,
            "log": {"action": "building", "player_id": player_id, "card_id": card_id, "color": rent_color},
        }


    raise ValueError("Unsupported action effect.")


In [37]:
## Start Action

def start_action(
    state: GameState,
    catalog: Dict[str, CardDef],
    player_id: str,
    *,
    action_type: str,
    card_id: Optional[str] = None,
    bank_card_id: Optional[str] = None,
    property_card_id: Optional[str] = None,
    property_color: Optional[str] = None,
    rent_color: Optional[str] = None,
    double_rent_ids: Optional[List[str]] = None,
    target_player_id: Optional[str] = None,
    steal_card_id: Optional[str] = None,
    give_card_id: Optional[str] = None,
    steal_color: Optional[str] = None,
    change_wild: Optional[Dict[str, str]] = None,
    discard_ids: Optional[List[str]] = None,
) -> Dict[str, Any]:
    try:
        if player_id not in state.players:
            raise ValueError("Unknown player_id.")
        actor = state.players[player_id]

        # === Play bank card ===
        if action_type == "play_bank":
            if not bank_card_id:
                raise ValueError("bank_card_id required.")
            if bank_card_id not in actor.hand:
                raise ValueError("Bank card not in hand.")
            use_action(state)
            play_bank(state, player_id, bank_card_id, catalog)
            return {"status": "ok", "response_type": "action_resolved", "state": state}

        # === Play property ===
        if action_type == "play_property":
            if not property_card_id:
                raise ValueError("property_card_id required.")
            if property_card_id not in actor.hand:
                raise ValueError("Property card not in hand.")
            use_action(state)
            play_property(state, catalog, player_id, property_card_id, property_color)
            return {"status": "ok", "response_type": "action_resolved", "state": state}

        # === Change wild ===
        if action_type == "change_wild":
            if not change_wild:
                raise ValueError("change_wild required.")
            # validate before consuming action
            if change_wild["card_id"] not in actor.hand and all(
                change_wild["card_id"] not in cards for cards in actor.properties.values()
            ):
                raise ValueError("Wild card not in properties.")
            use_action(state)
            change_wild_color(state, catalog, player_id, change_wild["card_id"], change_wild["new_color"])
            return {"status": "ok", "response_type": "action_resolved", "state": state}

        # === Discard / End turn ===
        if action_type == "discard":
            discard_cards(state, player_id, discard_ids)
            return {"status": "ok", "response_type": "action_resolved", "state": state}

        if action_type == "end_turn":
            return {"status": "ok", "response_type": "action_resolved", "state": end_turn(state, list(state.players.keys()))}

        # === Play action card ===
        if action_type in {"play_action_counterable", "play_action_non_counterable"}:
            if not card_id:
                raise ValueError("card_id required.")
            if card_id not in actor.hand:
                raise ValueError("Card not in hand.")
            card = catalog[card_id]

            if action_type == "play_action_counterable":
                if not is_counterable(card):
                    raise ValueError("Card is not counterable.")
                if not target_player_id:
                    raise ValueError("target_player_id required for counterable action.")
                # validation done → consume action
                use_action(state)

                discard_action_cards(state, actor, [card_id] + (double_rent_ids or []))
                pending_id = create_pending_action(
                    state,
                    source_player=player_id,
                    target_player=target_player_id,
                    card_id=card_id,
                    card_kind=card.kind,
                    effect=card.play.effect if card.play else None,
                    payload={
                        "rent_color": rent_color,
                        "double_rent_ids": double_rent_ids,
                        "steal_card_id": steal_card_id,
                        "give_card_id": give_card_id,
                        "steal_color": steal_color,
                        "amount": card.play.params.get("amount") if card.play else None,
                    },
                )

                return {
                    "status": "ok",
                    "response_type": "response_required",
                    "state": state,
                    "response_required": {
                        "pending_requests": [{
                            "pending_id": pending_id,
                            "target_player": target_player_id,
                            "prompt": "Accept or Just Say No?"
                        }]
                    },
                }

            # non‑counterable branch
            if is_counterable(card):
                raise ValueError("Card is counterable; use play_action_counterable.")

            # validation done → consume action
            use_action(state)

            return apply_action_effects(
                state,
                catalog,
                player_id=player_id,
                card_id=card_id,
                rent_color=rent_color,
                double_rent_ids=double_rent_ids,
                target_player_id=target_player_id,
                steal_card_id=steal_card_id,
                give_card_id=give_card_id,
                steal_color=steal_color,
                already_discarded=False,
            )

        raise ValueError("Unknown action_type.")

    except Exception as e:
        return {"status": "error", "response_type": "action_resolved", "message": str(e)}


In [39]:
## Respond to pending

def respond_to_pending(
    state: GameState,
    catalog: Dict[str, CardDef],
    pending_id: str,
    player_id: str,
    response: str,
) -> Dict[str, Any]:
    if pending_id not in state.pending_actions:
        return {"status": "error", "response_type": "action_resolved", "message": "Pending not found."}

    pending = state.pending_actions[pending_id]

    if player_id != pending["awaiting_player"]:
        return {"status": "error", "response_type": "action_resolved", "message": "Not your response turn."}

    if response == "accept":
        if pending["jsn_count"] % 2 == 1:
            del state.pending_actions[pending_id]
            return {"status": "ok", "response_type": "action_resolved", "state": state,
                    "log": {"action": "canceled_by_jsn", "pending_id": pending_id}}

        result = apply_action_effects(
            state,
            catalog,
            player_id=pending["source_player"],
            card_id=pending["card_id"],
            rent_color=pending["payload"].get("rent_color"),
            double_rent_ids=pending["payload"].get("double_rent_ids"),
            target_player_id=pending["target_player"],
            steal_card_id=pending["payload"].get("steal_card_id"),
            give_card_id=pending["payload"].get("give_card_id"),
            steal_color=pending["payload"].get("steal_color"),
            already_discarded=True,
        )
        del state.pending_actions[pending_id]
        return result

    if response == "just_say_no":
        if JSN_CARD_ID not in state.players[player_id].hand:
            return {"status": "error", "response_type": "action_resolved", "message": "No JSN in hand."}

        state.players[player_id].hand.remove(JSN_CARD_ID)
        state.deck.discard_pile.append(JSN_CARD_ID)

        pending["jsn_count"] += 1
        pending["awaiting_player"] = pending["source_player"] if player_id == pending["target_player"] else pending["target_player"]

        return {
            "status": "ok",
            "response_type": "response_required",
            "state": state,
            "response_required": {
                "pending_requests": [{
                    "pending_id": pending_id,
                    "target_player": pending["awaiting_player"],
                    "prompt": "Opponent played Just Say No. Counter?"
                }]
            },
        }

    return {"status": "error", "response_type": "action_resolved", "message": "Unknown response."}


In [ ]:
from collections import Counter
from typing import Dict, List, Any

def remove_ids_from_list(pool: List[str], ids_to_remove: List[str]) -> None:
    counts = Counter(pool)
    remove_counts = Counter(ids_to_remove)
    for cid, cnt in remove_counts.items():
        if cnt > counts.get(cid, 0):
            raise ValueError(f"Cannot remove {cnt} copies of {cid} from list.")
    for cid in ids_to_remove:
        pool.remove(cid)

def remove_ids_from_properties(properties: Dict[str, List[str]], ids_to_remove: List[str]) -> Dict[str, List[str]]:
    to_remove = Counter(ids_to_remove)
    removed_by_color: Dict[str, List[str]] = {}

    for color, cards in properties.items():
        new_cards = []
        for cid in cards:
            if to_remove[cid] > 0:
                to_remove[cid] -= 1
                removed_by_color.setdefault(color, []).append(cid)
            else:
                new_cards.append(cid)
        properties[color] = new_cards

    missing = [cid for cid, cnt in to_remove.items() if cnt > 0]
    if missing:
        raise ValueError(f"Card ids not found in properties: {missing}")

    return removed_by_color

def remove_ids_from_buildings(buildings: Dict[str, List[str]], ids_to_remove: List[str]) -> Dict[str, List[str]]:
    to_remove = Counter(ids_to_remove)
    removed_by_color: Dict[str, List[str]] = {}

    for color, cards in buildings.items():
        new_cards = []
        for cid in cards:
            if to_remove[cid] > 0:
                to_remove[cid] -= 1
                removed_by_color.setdefault(color, []).append(cid)
            else:
                new_cards.append(cid)
        buildings[color] = new_cards

    missing = [cid for cid, cnt in to_remove.items() if cnt > 0]
    if missing:
        raise ValueError(f"Building ids not found: {missing}")

    return removed_by_color

def card_value(cid: str, catalog: Dict[str, CardDef]) -> int:
    if cid not in catalog:
        raise ValueError(f"Unknown card id: {cid}")
    return catalog[cid].money_value

def process_payment(
    state: GameState,
    payer_id: str,
    receiver_id: str,
    catalog: Dict[str, CardDef],
    user_bank_payment_ids: List[str],
    user_property_payment_ids: List[str],
    user_building_payment_ids: List[str],
    money_charged: int,
) -> Dict[str, Any]:
    
    #TODO: Implement Payment of Hotel FIRST and then houses! But honestly we can do that later.
    try:
        if payer_id not in state.players:
            raise ValueError(f"Unknown payer_id: {payer_id}")
        if receiver_id not in state.players:
            raise ValueError(f"Unknown receiver_id: {receiver_id}")

        payer = state.players[payer_id]
        receiver = state.players[receiver_id]

        bank_counts = Counter(payer.bank)
        prop_counts = Counter([cid for cards in payer.properties.values() for cid in cards])
        bld_counts = Counter([cid for cards in payer.buildings.values() for cid in cards])

        bank_pay_counts = Counter(user_bank_payment_ids)
        prop_pay_counts = Counter(user_property_payment_ids)
        bld_pay_counts = Counter(user_building_payment_ids)

        for cid, cnt in bank_pay_counts.items():
            if cnt > bank_counts.get(cid, 0):
                raise ValueError(f"Payment includes {cid} not in bank or exceeds count.")

        for cid, cnt in prop_pay_counts.items():
            if cnt > prop_counts.get(cid, 0):
                raise ValueError(f"Payment includes {cid} not in properties or exceeds count.")

        for cid, cnt in bld_pay_counts.items():
            if cnt > bld_counts.get(cid, 0):
                raise ValueError(f"Payment includes {cid} not in buildings or exceeds count.")

        offered_total = (
            sum(card_value(cid, catalog) for cid in user_bank_payment_ids)
            + sum(card_value(cid, catalog) for cid in user_property_payment_ids)
            + sum(card_value(cid, catalog) for cid in user_building_payment_ids)
        )

        available_total = (
            sum(card_value(cid, catalog) for cid in payer.bank)
            + sum(card_value(cid, catalog) for cid in prop_counts.elements())
            + sum(card_value(cid, catalog) for cid in bld_counts.elements())
        )

        paid_all_bank = bank_pay_counts == bank_counts
        paid_all_props = prop_pay_counts == prop_counts
        paid_all_blds = bld_pay_counts == bld_counts

        if offered_total >= money_charged:
            remove_ids_from_list(payer.bank, user_bank_payment_ids)
            removed_props = remove_ids_from_properties(payer.properties, user_property_payment_ids)
            removed_blds = remove_ids_from_buildings(payer.buildings, user_building_payment_ids)
            fully_paid = True
        elif (available_total < money_charged) and (offered_total == available_total) and paid_all_bank and paid_all_props and paid_all_blds:
            remove_ids_from_list(payer.bank, user_bank_payment_ids)
            removed_props = remove_ids_from_properties(payer.properties, user_property_payment_ids)
            removed_blds = remove_ids_from_buildings(payer.buildings, user_building_payment_ids)
            fully_paid = False
        else:
            raise ValueError("Insufficient payment.")

        receiver.bank.extend(user_bank_payment_ids)
        for color, cards in removed_props.items():
            receiver.properties.setdefault(color, []).extend(cards)
        for color, cards in removed_blds.items():
            receiver.buildings.setdefault(color, []).extend(cards)

        return {
            "status": "ok",
            "response_type": "payment_applied",
            "state": state,
            "log": {
                "payer_id": payer_id,
                "receiver_id": receiver_id,
                "paid_amount": offered_total,
                "bank_cards": user_bank_payment_ids,
                "property_cards": user_property_payment_ids,
                "building_cards": user_building_payment_ids,
                "fully_paid": fully_paid,
            },
        }

    except Exception as e:
        return {"status": "error", "response_type": "payment_applied", "message": str(e)}
